# 从零开始的扩散模型

有时从最简单可能的版本入手，有助于更好地理解某样东西如何工作。在本笔记本中，我们将从「玩具」扩散模型开始，看看各个部分如何协作，然后考察它们与更复杂实现的差异。

我们将探讨
- 腐蚀过程（向数据添加噪声）
- UNet 是什么，以及如何从头实现一个极简版本
- 扩散模型训练
- 采样理论

然后我们将自己的版本与 diffusers 的 DDPM 实现对比，探索
- 相比我们的迷你 UNet 的改进
- DDPM 噪声调度
- 训练目标的差异
- 时间步条件（Timestep conditioning）
- 采样方法

本笔记本内容较深，如果你对从零开始的深入剖析不感兴趣，可以安全跳过！

还值得说明的是，此处大多数代码用于说明目的，我不建议直接将其中的任何部分用于你自己的工作（除非你只是为了学习而尝试改进这里展示的示例）。


## 环境准备与导入


In [ ]:
%pip install -q diffusers

In [ ]:
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from diffusers import DDPMScheduler, UNet2DModel
from matplotlib import pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

## 数据

这里我们将用非常小的数据集 mnist 来测试。如果你想给模型稍难一点的挑战而不改其他任何东西，`torchvision.datasets.FashionMNIST` 可以作为即插即用的替代。


In [ ]:
dataset = torchvision.datasets.MNIST(root="mnist/", train=True, download=True, transform=torchvision.transforms.ToTensor())

In [ ]:
train_dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

In [ ]:
x, y = next(iter(train_dataloader))
print('Input shape:', x.shape)
print('Labels:', y)
plt.imshow(torchvision.utils.make_grid(x)[0], cmap='Greys');

每张图像是 28×28 像素的灰度数字手绘，取值范围为 0 到 1。


## 腐蚀过程

假设你还没读过任何扩散模型论文，但知道过程涉及添加噪声。你会怎么做？

我们可能希望有一种简单的方法来控制腐蚀程度。那么，如果引入一个参数表示要添加的 `amount` 噪声量，然后执行：

`noise = torch.rand_like(x)` 

`noisy_x =  (1-amount)*x + amount*noise`

当 amount = 0 时，我们得到未经改变的输入。当 amount 达到 1 时，我们得到纯噪声，输入 x 的痕迹荡然无存。以这种方式将输入与噪声混合，可使输出保持在相同范围（0 到 1）。

我们可以相当容易地实现这一点（注意形状，避免被广播规则坑到）：


In [ ]:
def corrupt(x, amount):
  """Corrupt the input `x` by mixing it with noise according to `amount`"""
  noise = torch.rand_like(x)
  amount = amount.view(-1, 1, 1, 1) # Sort shape so broadcasting works
  return x*(1-amount) + noise*amount 

并通过可视化结果来确认其按预期工作：


In [ ]:
# Plotting the input data
fig, axs = plt.subplots(2, 1, figsize=(12, 5))
axs[0].set_title('Input data')
axs[0].imshow(torchvision.utils.make_grid(x)[0], cmap='Greys')

# Adding noise
amount = torch.linspace(0, 1, x.shape[0]) # Left to right -> more corruption
noised_x = corrupt(x, amount)

# Plotting the noised version
axs[1].set_title('Corrupted data (-- amount increases -->)')
axs[1].imshow(torchvision.utils.make_grid(noised_x)[0], cmap='Greys');

当噪声量接近 1 时，数据开始看起来像纯随机噪声。但在大多数噪声量下，你仍能相当好地猜出数字。你认为这是最优的吗？


## 模型

我们希望有一个模型：输入 28 像素的带噪图像，输出相同形状的预测。这里常用的架构是 UNet。[最初为医学影像分割任务而发明](https://arxiv.org/abs/1505.04597)，UNet 由一条「收缩路径」（数据被压缩）和一条「扩展路径」（数据扩展回原始维度，类似自编码器）组成，同时还具有跳跃连接，使信息和梯度能在不同层级流动。

一些 UNet 在每个阶段使用复杂的块，但在这个玩具演示中，我们将构建一个极简示例：输入单通道图像，在下路径（图中的 down_layers 和代码中）经过三个卷积层，在上路径经过三个，下采样层与上采样层之间有跳跃连接。我们使用最大池化进行下采样，用 `nn.Upsample` 进行上采样，而不依赖更复杂 UNet 中的可学习层。下面是粗略架构，显示每层输出的通道数：


![unet_diag.png](../images/unit1/unet_diag.png)



代码实现大致如下：


In [ ]:
class BasicUNet(nn.Module):
    """A minimal UNet implementation."""
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.down_layers = torch.nn.ModuleList([ 
            nn.Conv2d(in_channels, 32, kernel_size=5, padding=2),
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
        ])
        self.up_layers = torch.nn.ModuleList([
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
            nn.Conv2d(64, 32, kernel_size=5, padding=2),
            nn.Conv2d(32, out_channels, kernel_size=5, padding=2), 
        ])
        self.act = nn.SiLU() # The activation function
        self.downscale = nn.MaxPool2d(2)
        self.upscale = nn.Upsample(scale_factor=2)

    def forward(self, x):
        h = []
        for i, l in enumerate(self.down_layers):
            x = self.act(l(x)) # Through the layer and the activation function
            if i < 2: # For all but the third (final) down layer:
              h.append(x) # Storing output for skip connection
              x = self.downscale(x) # Downscale ready for the next layer
              
        for i, l in enumerate(self.up_layers):
            if i > 0: # For all except the first up layer
              x = self.upscale(x) # Upscale
              x += h.pop() # Fetching stored output (skip connection)
            x = self.act(l(x)) # Through the layer and the activation function
            
        return x

我们可以验证输出形状与输入相同，符合预期：


In [ ]:
net = BasicUNet()
x = torch.rand(8, 1, 28, 28)
net(x).shape

该网络参数量略超过 30 万：


In [ ]:
sum([p.numel() for p in net.parameters()])

你可以尝试改变每层的通道数，或换用不同架构。


## 训练网络

那么模型到底应该做什么？同样有多种观点，但在这个演示中，我们采用简单的表述：给定腐蚀后的输入 noisy_x，模型应输出对原始 x 的最佳猜测。我们通过均方误差与真实值比较。

现在可以尝试训练网络。
- 获取一批数据
- 以随机强度腐蚀
- 送入模型
- 将模型预测与干净图像比较以计算损失
- 相应更新模型参数

欢迎修改并看看能否取得更好效果！


In [ ]:
# Dataloader (you can mess with batch size)
batch_size = 128
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# How many runs through the data should we do?
n_epochs = 3

# Create the network
net = BasicUNet()
net.to(device)

# Our loss function
loss_fn = nn.MSELoss()

# The optimizer
opt = torch.optim.Adam(net.parameters(), lr=1e-3) 

# Keeping a record of the losses for later viewing
losses = []

# The training loop
for epoch in range(n_epochs):

    for x, y in train_dataloader:

        # Get some data and prepare the corrupted version
        x = x.to(device) # Data on the GPU
        noise_amount = torch.rand(x.shape[0]).to(device) # Pick random noise amounts
        noisy_x = corrupt(x, noise_amount) # Create our noisy x

        # Get the model prediction
        pred = net(noisy_x)

        # Calculate the loss
        loss = loss_fn(pred, x) # How close is the output to the true 'clean' x?

        # Backprop and update the params:
        opt.zero_grad()
        loss.backward()
        opt.step()

        # Store the loss for later
        losses.append(loss.item())

    # Print our the average of the loss values for this epoch:
    avg_loss = sum(losses[-len(train_dataloader):])/len(train_dataloader)
    print(f'Finished epoch {epoch}. Average loss for this epoch: {avg_loss:05f}')

# View the loss curve
plt.plot(losses)
plt.ylim(0, 0.1);

我们可以取一批数据，以不同强度腐蚀，然后查看模型预测，了解预测效果：


In [ ]:
#@markdown Visualizing model predictions on noisy inputs:

# Fetch some data
x, y = next(iter(train_dataloader))
x = x[:8] # Only using the first 8 for easy plotting

# Corrupt with a range of amounts
amount = torch.linspace(0, 1, x.shape[0]) # Left to right -> more corruption
noised_x = corrupt(x, amount)

# Get the model predictions
with torch.no_grad():
  preds = net(noised_x.to(device)).detach().cpu()

# Plot
fig, axs = plt.subplots(3, 1, figsize=(12, 7))
axs[0].set_title('Input data')
axs[0].imshow(torchvision.utils.make_grid(x)[0].clip(0, 1), cmap='Greys')
axs[1].set_title('Corrupted data')
axs[1].imshow(torchvision.utils.make_grid(noised_x)[0].clip(0, 1), cmap='Greys')
axs[2].set_title('Network Predictions')
axs[2].imshow(torchvision.utils.make_grid(preds)[0].clip(0, 1), cmap='Greys');

可以看到，在较低噪声量时预测相当不错！但当噪声水平很高时，模型可利用的信息更少，到 amount=1 时，它输出接近数据集均值的模糊结果，试图在对输出可能长什么样上「两边下注」……


## 采样

如果我们在高噪声水平上的预测不太好，如何生成图像？

那么，如果从随机噪声开始，查看模型预测，但只向该预测移动一小步——比如说，走 20% 的路。现在我们有一张非常带噪的图像，其中或许有一点结构，可以送入模型得到新预测。希望这个新预测比第一个稍好（因为起点噪声稍少），于是我们可以用这个新的、更好的预测再迈一小步。

重复几次，（如果一切顺利）就能得到图像！下面仅用 5 步说明该过程，可视化每阶段的模型输入（左）和预测的去噪图像（右）。注意，尽管模型在第 1 步就预测出去噪图像，我们仍只将 x 向该方向移动一部分。经过几步，结构出现并 refine，直到得到最终输出。


In [ ]:
#@markdown Sampling strategy: Break the process into 5 steps and move 1/5'th of the way there each time:
n_steps = 5
x = torch.rand(8, 1, 28, 28).to(device) # Start from random
step_history = [x.detach().cpu()]
pred_output_history = []

for i in range(n_steps):
    with torch.no_grad(): # No need to track gradients during inference
        pred = net(x) # Predict the denoised x0
    pred_output_history.append(pred.detach().cpu()) # Store model output for plotting
    mix_factor = 1/(n_steps - i) # How much we move towards the prediction
    x = x*(1-mix_factor) + pred*mix_factor # Move part of the way there
    step_history.append(x.detach().cpu()) # Store step for plotting

fig, axs = plt.subplots(n_steps, 2, figsize=(9, 4), sharex=True)
axs[0,0].set_title('x (model input)')
axs[0,1].set_title('model prediction')
for i in range(n_steps):
    axs[i, 0].imshow(torchvision.utils.make_grid(step_history[i])[0].clip(0, 1), cmap='Greys')
    axs[i, 1].imshow(torchvision.utils.make_grid(pred_output_history[i])[0].clip(0, 1), cmap='Greys')

我们可以将过程拆成更多步骤，希望由此得到更好的图像：


In [ ]:
#@markdown Showing more results, using 40 sampling steps
n_steps = 40
x = torch.rand(64, 1, 28, 28).to(device)
for i in range(n_steps):
  noise_amount = torch.ones((x.shape[0], )).to(device) * (1-(i/n_steps)) # Starting high going low
  with torch.no_grad():
    pred = net(x)
  mix_factor = 1/(n_steps - i)
  x = x*(1-mix_factor) + pred*mix_factor
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
ax.imshow(torchvision.utils.make_grid(x.detach().cpu(), nrow=8)[0].clip(0, 1), cmap='Greys')

效果不算好，但已经能辨认出一些数字！你可以尝试训练更久（比如 10 或 20 个 epoch），并调整模型配置、学习率、优化器等。另外，如果想试稍难的数据集，别忘了 fashionMNIST 只需一行替换。


## 与 DDPM 的对比

本节我们将看看玩具实现与另一笔记本（[Diffusers 入门](https://github.com/huggingface/diffusion-models-class/blob/main/unit1/01_introduction_to_diffusers.ipynb)）中基于 DDPM 论文的方法有何不同。

我们将看到


*   diffusers 的 `UNet2DModel` 比我们的 BasicUNet 更先进
*   腐蚀过程的处理方式不同
*   训练目标不同，涉及预测噪声而非去噪图像
*   模型通过时间步条件了解当前噪声量，t 作为额外参数传入 forward 方法
*   有多种不同的采样策略可用，应比上面简陋的版本效果更好

自 DDPM 论文发表以来已有许多改进建议，但本示例希望能说明各种可用的设计决策。阅读完本节后，你可能有兴趣深入阅读论文 ['Elucidating the Design Space of Diffusion-Based Generative Models'](https://arxiv.org/abs/2206.00364)，它详细探讨了所有这些组件并提出了获得最佳性能的新建议。

如果这一切过于技术化或令人望而生畏，别担心！可以随意跳过本笔记本的剩余部分，或留到有空时再读。





### UNet

diffusers 的 UNet2DModel 相比上面的基础 UNet 有多项改进：

*   GroupNorm 对每个块的输入应用组归一化
*   Dropout 层使训练更平滑
*   每个块多个 resnet 层（若 layers_per_block 未设为 1）
*   注意力（通常仅用于较低分辨率块）
*   对时间步的条件化
*   带可学习参数的下采样和上采样块

让我们创建并检查一个 UNet2DModel：




In [ ]:
model = UNet2DModel(
    sample_size=28,           # the target image resolution
    in_channels=1,            # the number of input channels, 3 for RGB images
    out_channels=1,           # the number of output channels
    layers_per_block=2,       # how many ResNet layers to use per UNet block
    block_out_channels=(32, 64, 64), # Roughly matching our basic unet example
    down_block_types=( 
        "DownBlock2D",        # a regular ResNet downsampling block
        "AttnDownBlock2D",    # a ResNet downsampling block with spatial self-attention
        "AttnDownBlock2D",
    ), 
    up_block_types=(
        "AttnUpBlock2D", 
        "AttnUpBlock2D",      # a ResNet upsampling block with spatial self-attention
        "UpBlock2D",          # a regular ResNet upsampling block
      ),
)
print(model)

如你所见，复杂得多！参数量也显著多于我们的 BasicUNet：


In [ ]:
sum([p.numel() for p in model.parameters()]) # 1.7M vs the ~309k parameters of the BasicUNet

我们可以用该模型替代原来的模型，复现上面的训练。需要同时传入 x 和 timestep（这里我始终传入 t=0，以表明在没有时间步条件化时也能工作，并保持采样代码简单，但你也可以尝试传入 `(amount*1000)` 从腐蚀量得到等价时间步）。改动的行用 `#<<<` 标记，便于检查代码。


In [ ]:
#@markdown Trying UNet2DModel instead of BasicUNet:

# Dataloader (you can mess with batch size)
batch_size = 128
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# How many runs through the data should we do?
n_epochs = 3

# Create the network
net = UNet2DModel(
    sample_size=28,  # the target image resolution
    in_channels=1,  # the number of input channels, 3 for RGB images
    out_channels=1,  # the number of output channels
    layers_per_block=2,  # how many ResNet layers to use per UNet block
    block_out_channels=(32, 64, 64),  # Roughly matching our basic unet example
    down_block_types=( 
        "DownBlock2D",  # a regular ResNet downsampling block
        "AttnDownBlock2D",  # a ResNet downsampling block with spatial self-attention
        "AttnDownBlock2D",
    ), 
    up_block_types=(
        "AttnUpBlock2D", 
        "AttnUpBlock2D",  # a ResNet upsampling block with spatial self-attention
        "UpBlock2D",   # a regular ResNet upsampling block
      ),
) #<<<
net.to(device)

# Our loss finction
loss_fn = nn.MSELoss()

# The optimizer
opt = torch.optim.Adam(net.parameters(), lr=1e-3) 

# Keeping a record of the losses for later viewing
losses = []

# The training loop
for epoch in range(n_epochs):

    for x, y in train_dataloader:

        # Get some data and prepare the corrupted version
        x = x.to(device) # Data on the GPU
        noise_amount = torch.rand(x.shape[0]).to(device) # Pick random noise amounts
        noisy_x = corrupt(x, noise_amount) # Create our noisy x

        # Get the model prediction
        pred = net(noisy_x, 0).sample #<<< Using timestep 0 always, adding .sample

        # Calculate the loss
        loss = loss_fn(pred, x) # How close is the output to the true 'clean' x?

        # Backprop and update the params:
        opt.zero_grad()
        loss.backward()
        opt.step()

        # Store the loss for later
        losses.append(loss.item())

    # Print our the average of the loss values for this epoch:
    avg_loss = sum(losses[-len(train_dataloader):])/len(train_dataloader)
    print(f'Finished epoch {epoch}. Average loss for this epoch: {avg_loss:05f}')

# Plot losses and some samples
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Losses
axs[0].plot(losses)
axs[0].set_ylim(0, 0.1)
axs[0].set_title('Loss over time')

# Samples
n_steps = 40
x = torch.rand(64, 1, 28, 28).to(device)
for i in range(n_steps):
  noise_amount = torch.ones((x.shape[0], )).to(device) * (1-(i/n_steps)) # Starting high going low
  with torch.no_grad():
    pred = net(x, 0).sample
  mix_factor = 1/(n_steps - i)
  x = x*(1-mix_factor) + pred*mix_factor

axs[1].imshow(torchvision.utils.make_grid(x.detach().cpu(), nrow=8)[0].clip(0, 1), cmap='Greys')
axs[1].set_title('Generated Samples');

这比第一组结果好不少！你可以尝试调整 unet 配置或训练更久以获得更好性能。


### 腐蚀过程

DDPM 论文描述了一个腐蚀过程，在每个「时间步」添加少量噪声。给定某个时间步的 $x_{t-1}$，我们可以得到下一个（噪声稍大一点的）版本 $x_t$：<br><br>

$q(\mathbf{x}_t \vert \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t; \sqrt{1 - \beta_t} \mathbf{x}_{t-1}, \beta_t\mathbf{I}) \quad
q(\mathbf{x}_{1:T} \vert \mathbf{x}_0) = \prod^T_{t=1} q(\mathbf{x}_t \vert \mathbf{x}_{t-1})$<br><br>


也就是说，我们取 $x_{t-1}$，乘以 $\sqrt{1 - \beta_t}$，再加上按 $\beta_t$ 缩放的噪声。这个 $\beta$ 根据某种调度为每个 t 定义，并决定每个时间步添加多少噪声。我们不一定想执行 500 次该操作来得到 $x_{500}$，因此还有另一个公式，给定 $x_0$ 直接得到任意 t 的 $x_t$：<br><br>

$\begin{aligned}
q(\mathbf{x}_t \vert \mathbf{x}_0) &= \mathcal{N}(\mathbf{x}_t; \sqrt{\bar{\alpha}_t} \mathbf{x}_0, \sqrt{(1 - \bar{\alpha}_t)} \mathbf{I})
\end{aligned}$ 其中 $\bar{\alpha}_t = \prod_{i=1}^T \alpha_i$，且 $\alpha_i = 1-\beta_i$<br><br>

数学符号总是看起来很吓人！好在调度器会帮我们处理这一切（取消注释下一个单元格可查看代码）。我们可以绘制 $\sqrt{\bar{\alpha}_t}$（标记为 `sqrt_alpha_prod`）和 $\sqrt{(1 - \bar{\alpha}_t)}$（标记为 `sqrt_one_minus_alpha_prod`），查看输入 (x) 与噪声在不同时间步如何被缩放和混合：



In [ ]:
#??noise_scheduler.add_noise

In [ ]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
plt.plot(noise_scheduler.alphas_cumprod.cpu() ** 0.5, label=r"${\sqrt{\bar{\alpha}_t}}$")
plt.plot((1 - noise_scheduler.alphas_cumprod.cpu()) ** 0.5, label=r"$\sqrt{(1 - \bar{\alpha}_t)}$")
plt.legend(fontsize="x-large");

起初，带噪 x 主要是 x（sqrt_alpha_prod ≈ 1），但随着时间推移，x 的贡献下降，噪声成分增加。与按 `amount` 线性混合 x 和噪声不同，这种方式相对更快地变噪。我们可以在一些数据上可视化这一点：


In [ ]:
#@markdown visualize the DDPM noising process for different timesteps:

# Noise a batch of images to view the effect
fig, axs = plt.subplots(3, 1, figsize=(16, 10))
xb, yb = next(iter(train_dataloader))
xb = xb.to(device)[:8]
xb = xb * 2. - 1. # Map to (-1, 1)
print('X shape', xb.shape)

# Show clean inputs
axs[0].imshow(torchvision.utils.make_grid(xb[:8])[0].detach().cpu(), cmap='Greys')
axs[0].set_title('Clean X')

# Add noise with scheduler
timesteps = torch.linspace(0, 999, 8).long().to(device)
noise = torch.randn_like(xb) # << NB: randn not rand
noisy_xb = noise_scheduler.add_noise(xb, noise, timesteps)
print('Noisy X shape', noisy_xb.shape)

# Show noisy version (with and without clipping)
axs[1].imshow(torchvision.utils.make_grid(noisy_xb[:8])[0].detach().cpu().clip(-1, 1),  cmap='Greys')
axs[1].set_title('Noisy X (clipped to (-1, 1)')
axs[2].imshow(torchvision.utils.make_grid(noisy_xb[:8])[0].detach().cpu(),  cmap='Greys')
axs[2].set_title('Noisy X');

还有另一个动态：DDPM 版本添加的是从高斯分布（torch.randn，均值 0、标准差 1）抽取的噪声，而不是我们原始 `corrupt` 函数中 0 到 1 的均匀噪声（torch.rand）。一般来说，归一化训练数据也有意义。在另一笔记本中，你会在变换列表中看到 `Normalize(0.5, 0.5)`，将图像数据从 (0, 1) 映射到 (-1, 1)，对我们的目的「足够好」。本笔记本未这样做，但上面的可视化单元格为更准确的缩放和可视化加入了归一化。


### 训练目标

在我们的玩具示例中，让模型尝试预测去噪后的图像。在 DDPM 和许多其他扩散模型实现中，模型预测腐蚀过程中使用的噪声（缩放前，即单位方差噪声）。代码大致如下：

```python
noise = torch.randn_like(xb) # << NB: randn not rand
noisy_x = noise_scheduler.add_noise(x, noise, timesteps)
model_prediction = model(noisy_x, timesteps).sample
loss = mse_loss(model_prediction, noise) # noise as the target
```

你可能认为预测噪声（从中可推出去噪图像的样子）与直接预测去噪图像等价。那么为何偏爱其一——仅仅是为了数学便利吗？

这里还有一个微妙之处。我们在训练期间在不同（随机选择的）时间步上计算损失。这些不同目标会导致对这些损失的不同「隐式加权」，预测噪声会更重视较低噪声水平。你可以选择更复杂的目标来改变这种「隐式损失加权」。或者选择一种噪声调度，使较高噪声水平的样本更多。也许让模型预测「速度」v，我们将其定义为根据噪声水平由图像和噪声组合而成（见 'PROGRESSIVE DISTILLATION FOR FAST SAMPLING OF DIFFUSION MODELS'）。也许让模型预测噪声，然后根据少量理论按噪声量缩放损失（见 'Perception Prioritized Training of Diffusion Models'），或根据实验看哪些噪声水平对模型最有信息量（见 'Elucidating the Design Space of Diffusion-Based Generative Models'）。简而言之：选择目标会影响模型性能，相关研究仍在进行。

目前，预测噪声（epsilon 或 eps，你会在某些地方看到）是主流方法，但随着时间推移，库中可能会支持其他目标并在不同情况下使用。




### 时间步条件

UNet2DModel 同时接收 x 和 timestep。后者被转换为嵌入，并在模型多处注入。

其理论依据是：向模型提供噪声水平信息，能更好地完成任务。虽然可以在没有时间步条件的情况下训练模型，但在某些情况下它似乎有助于性能，当前文献中的大多数实现都包含它。


### 采样

给定一个估计带噪输入中噪声（或预测去噪版本）的模型，如何生成新图像？

我们可以输入纯噪声，希望模型一步预测出良好的去噪图像。然而，如上实验所示，这通常效果不佳。因此，我们根据模型预测采取多步较小步长，迭代地去除少量噪声。

具体如何迈步取决于所使用的采样方法。我们不会深入理论，但一些关键设计问题是：
- 步长应多大？换句话说，应遵循怎样的「噪声调度」？
- 是否仅使用模型当前预测来指导更新步（如 DDPM、DDIM 及许多其他方法）？是否多次评估模型以估计高阶梯度从而迈出更大、更准确的步（高阶方法及某些离散 ODE 求解器）？还是保留过去预测的历史以更好地指导当前更新（线性多步和 ancestral 采样器）？
- 是否添加额外噪声（有时称为 churn）为采样过程引入更多随机性，还是保持完全确定性？许多采样器用参数（如 DDIM 采样器的 'eta'）控制，供用户选择。

扩散模型采样方法的研究发展迅速，越来越多方法被提出，以在更少步数内找到良好解。勇敢且好奇的读者可能会觉得浏览 diffusers 库中[此处](https://github.com/huggingface/diffusers/tree/main/src/diffusers/schedulers)的不同实现，或查看常链接到相关论文的[文档](https://huggingface.co/docs/diffusers/api/schedulers/overview) 很有趣。


## 结论

希望这能从略有不同的角度帮助你理解扩散模型。

本笔记本由 Jonathan Whitaker 为本 Hugging Face 课程撰写，与其[自己课程中的版本](https://johnowhitaker.github.io/tglcourse/dm1.html)（《The Generative Landscape》）有重叠。如果你想看加入噪声和类别条件的基础示例扩展，可以去看看。问题或 bug 可通过 GitHub issue 或 Discord 反馈。也欢迎通过 Twitter [@johnowhitaker](https://twitter.com/johnowhitaker) 联系。
